# <font color="#418FDE" size="6.5" uppercase>**SVR Kernel Ridge**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Fit SVR variants for nonlinear and linear regression tasks. 
- Use One-Class SVM and precomputed kernels in appropriate workflows. 
- Compare SVR and Kernel Ridge in accuracy, smoothness, and computational cost. 


## **1. SVR Models**

### **1.1. Epsilon SVR**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_01_01.jpg?v=1783999446" width="250">



>* Predicts continuous values with an error tolerance tube
>* Focuses on meaningful deviations, reducing noise impact

>* Kernels fit linear or curved patterns
>* Support vectors focus on informative observations

>* Tune epsilon, regularization, and kernel settings
>* Scale features and validate for robust predictions



In [ ]:
#@title Python Code - Epsilon SVR

# This example fits epsilon support vector regression.
# The epsilon tube ignores small prediction errors.
# The plot shows support vectors and predictions.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error

# Create a small nonlinear regression dataset.
rng = np.random.default_rng(42)
x_values = np.linspace(0, 6, 80)
noise = rng.normal(0, 0.18, size=x_values.shape)
y_values = np.sin(x_values) + 0.25 * x_values + noise

# Validate the feature and target shapes.
X = x_values.reshape(-1, 1)
if X.shape[0] != y_values.shape[0]:
    raise ValueError("Feature and target lengths must match.")

# Fit one epsilon SVR model with an RBF kernel.
epsilon = 0.20
model = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=10, epsilon=epsilon))
model.fit(X, y_values)

# Predict smoothly across the input range.
x_grid = np.linspace(0, 6, 300).reshape(-1, 1)
y_pred = model.predict(x_grid)
y_train_pred = model.predict(X)

# Count points outside the epsilon tube.
residuals = np.abs(y_values - y_train_pred)
outside_count = int(np.sum(residuals > epsilon))
mae = mean_absolute_error(y_values, y_train_pred)

print("scikit-learn version:", sklearn.__version__)
print("Training MAE:", round(mae, 3))
print("Points outside epsilon tube:", outside_count)

# Draw the fitted curve and epsilon tolerance tube.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_values, y_values, s=28, alpha=0.75, label="training data")
ax.plot(x_grid.ravel(), y_pred, color="black", label="SVR prediction")
ax.fill_between(x_grid.ravel(), y_pred - epsilon, y_pred + epsilon, alpha=0.2, label="epsilon tube")
ax.set_title("Epsilon SVR focuses on errors outside the tube")
ax.set_xlabel("Input feature")
ax.set_ylabel("Target value")
ax.legend()
plt.show()



### **1.2. Linear SVR**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_01_02.jpg?v=1783999444" width="250">



>* Fits straight-line trends while ignoring small errors
>* Balances simple, interpretable predictions with accuracy

>* Tolerance zone ignores small prediction errors
>* Regularization balances stability and training accuracy

>* Efficient for high-dimensional linear data
>* Requires scaling; useful baseline before complexity



In [ ]:
#@title Python Code - Linear SVR

# This example fits a linear support vector regressor.
# Scaling helps SVR compare features fairly.
# The plot shows predictions and the tolerance band.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from sklearn.metrics import mean_absolute_error

# Create a small noisy linear regression dataset.
rng = np.random.default_rng(42)
area = np.linspace(30, 120, 80)
noise = rng.normal(0, 90, size=area.shape)

rent = 450 + 18 * area + noise
X = area.reshape(-1, 1)
y = rent

# Check that the feature matrix and target match.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target values.")

# Fit scaling and LinearSVR together in one pipeline.
model = make_pipeline(
    StandardScaler(),
    LinearSVR(C=1.0, epsilon=80.0, max_iter=10000, random_state=42)
)

model.fit(X, y)
predicted_rent = model.predict(X)
mae = mean_absolute_error(y, predicted_rent)

# Count points inside the epsilon tolerance zone.
absolute_errors = np.abs(y - predicted_rent)
inside_band = np.mean(absolute_errors <= 80.0) * 100

print("scikit-learn version:", sklearn.__version__)
print("Mean absolute error:", round(mae, 1), "dollars")
print("Within epsilon band:", round(inside_band, 1), "%")

# Plot the fitted line and its epsilon tolerance band.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(area, rent, label="training apartments", alpha=0.75)
ax.plot(area, predicted_rent, color="black", label="LinearSVR prediction")

ax.plot(area, predicted_rent + 80.0, color="red", linestyle="--", label="epsilon band")
ax.plot(area, predicted_rent - 80.0, color="red", linestyle="--")
ax.set_title("Linear SVR ignores small errors inside epsilon")
ax.set_xlabel("Apartment area (square meters)")

ax.set_ylabel("Monthly rent (dollars)")
ax.legend()
plt.show()



### **1.3. NuSVR Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_01_03.jpg?v=1783999448" width="250">



>* Nu controls support vectors and tolerated errors
>* Useful when error tolerance is unclear

>* Choose kernels for linear or curved patterns
>* Tune regularization and nu for flexibility

>* Controls sparsity when epsilon tuning feels unclear
>* Scale features and validate tuning choices



In [ ]:
#@title Python Code - NuSVR Regression

# This example fits NuSVR to nonlinear data.
# Scaling helps support vector models compare features fairly.
# The plot shows predictions and support vectors.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import NuSVR
from sklearn.metrics import mean_absolute_error

# Create a small curved regression dataset.
rng = np.random.default_rng(42)
x_values = np.linspace(0, 10, 120)
noise = rng.normal(0, 0.25, size=x_values.shape)
y_values = np.sin(x_values) + 0.15 * x_values + noise

# NuSVR expects a two-dimensional feature matrix.
features = x_values.reshape(-1, 1)
target = y_values

# Check the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and target must have matching rows.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42
)

# Scale features, then fit a nonlinear NuSVR model.
model = make_pipeline(
    StandardScaler(), NuSVR(kernel="rbf", C=10.0, nu=0.35, gamma="scale")
)
model.fit(X_train, y_train)

# Predict on the test set and a smooth plotting grid.
test_predictions = model.predict(X_test)
x_grid = np.linspace(0, 10, 300).reshape(-1, 1)
y_grid = model.predict(x_grid)

# Count support vectors used by the fitted NuSVR step.
nu_svr_step = model.named_steps["nusvr"]
support_count = len(nu_svr_step.support_)
support_fraction = support_count / len(X_train)

# Print concise results for the fitted model.
mae = mean_absolute_error(y_test, test_predictions)
print("scikit-learn version:", sklearn.__version__)
print("Test MAE:", round(mae, 3))
print("Training support vector fraction:", round(support_fraction, 2))

# Plot data, fitted curve, and support vectors.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train[:, 0], y_train, s=25, alpha=0.55, label="training data")
ax.scatter(X_test[:, 0], y_test, s=35, alpha=0.8, label="test data")
ax.plot(x_grid[:, 0], y_grid, color="black", linewidth=2, label="NuSVR curve")

# Highlight the observations that shape the NuSVR solution.
support_points = X_train[nu_svr_step.support_, 0]
support_targets = y_train[nu_svr_step.support_]
ax.scatter(
    support_points, support_targets, s=90, facecolors="none",
    edgecolors="red", label="support vectors"
)

# Label the single teaching plot clearly.
ax.set_title("NuSVR regression with an RBF kernel")
ax.set_xlabel("Input feature")
ax.set_ylabel("Continuous target")
ax.legend()
plt.show()



## **2. Kernel Workflows**

### **2.1. Anomaly Detection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_02_01.jpg?v=1783999439" width="250">



>* Learn normal patterns from mostly ordinary data
>* Flag observations outside the learned boundary

>* Kernels model complex normal behavior shapes
>* Kernel settings control anomaly boundary tightness

>* Use clean, scaled data for normality learning
>* Review alerts against costs and domain knowledge



In [ ]:
#@title Python Code - Anomaly Detection

# This example detects unusual points with One-Class SVM.
# The kernel learns a curved normal region.
# The plot shows normal and flagged observations.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

# Generate mostly normal data plus a few unusual points.
rng = np.random.default_rng(42)
angles = rng.uniform(0, 2 * np.pi, 180)
radii = rng.normal(2.0, 0.18, 180)

normal_points = np.column_stack((radii * np.cos(angles), radii * np.sin(angles)))
anomaly_points = rng.uniform(-4.0, 4.0, size=(20, 2))
all_points = np.vstack((normal_points, anomaly_points))

# Validate the small two-feature dataset before modeling.
if all_points.shape != (200, 2):
    raise ValueError("Expected 200 rows and 2 features.")

# Scale features because kernel distances depend on feature size.
scaler = StandardScaler()
scaled_normal = scaler.fit_transform(normal_points)
scaled_all = scaler.transform(all_points)

# Fit only on normal examples to learn typical behavior.
model = OneClassSVM(kernel="rbf", gamma=0.8, nu=0.08)
model.fit(scaled_normal)

# Predict returns 1 for normal and -1 for anomalies.
predictions = model.predict(scaled_all)
flagged_count = int(np.sum(predictions == -1))

print("scikit-learn version:", sklearn.__version__)
print("Training examples treated as normal:", len(normal_points))
print("Flagged observations after scoring all points:", flagged_count)

# Plot the learned anomaly decisions on the original feature scale.
colors = np.where(predictions == -1, "crimson", "steelblue")
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(all_points[:, 0], all_points[:, 1], c=colors, s=35, edgecolor="white")

ax.set_title("One-Class SVM anomaly detection with an RBF kernel")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.scatter([], [], c="steelblue", label="Predicted normal")
ax.scatter([], [], c="crimson", label="Flagged anomaly")

ax.legend(loc="upper right")
plt.show()



### **2.2. Kernel Ridge**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_02_02.jpg?v=1783999441" width="250">



>* Combines ridge stability with kernel flexibility
>* Learns smooth nonlinear patterns from similarities

>* Uses all data for smooth nonlinear predictions
>* Can become costly with large datasets

>* Learn from custom similarity matrices
>* Kernel quality drives prediction success



In [ ]:
#@title Python Code - Kernel Ridge

# Demonstrate Kernel Ridge with a precomputed kernel.
# Similarity matrices can replace raw feature tables.
# The plot compares true and predicted curves.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.metrics import mean_absolute_error

# Create a small nonlinear regression dataset.
rng = np.random.default_rng(42)
x_train = np.linspace(0, 6, 45).reshape(-1, 1)
y_train = np.sin(x_train).ravel() + rng.normal(0, 0.12, 45)

# Validate the shapes before building similarities.
if x_train.shape[0] != y_train.shape[0]:
    raise ValueError("Feature rows must match target values.")

# Build pairwise similarities for training observations.
gamma_value = 1.2
train_kernel = rbf_kernel(x_train, x_train, gamma=gamma_value)

# Fit Kernel Ridge directly on the similarity matrix.
model = KernelRidge(alpha=0.15, kernel="precomputed")
model.fit(train_kernel, y_train)

# Predict by comparing new points with training points.
x_test = np.linspace(0, 6, 200).reshape(-1, 1)
test_kernel = rbf_kernel(x_test, x_train, gamma=gamma_value)
y_pred = model.predict(test_kernel)

# Compare predictions with the noiseless target curve.
y_true = np.sin(x_test).ravel()
mae = mean_absolute_error(y_true, y_pred)
print("scikit-learn version:", sklearn.__version__)
print("Training kernel shape:", train_kernel.shape)
print("Mean absolute error versus smooth sine:", round(mae, 3))

# Plot the learned smooth function from precomputed similarities.
plt.figure(figsize=(7, 4))
plt.scatter(x_train.ravel(), y_train, label="noisy training data", s=28)
plt.plot(x_test.ravel(), y_true, label="true smooth curve", linewidth=2)
plt.plot(x_test.ravel(), y_pred, label="Kernel Ridge prediction", linewidth=2)
plt.title("Kernel Ridge using a precomputed RBF kernel")
plt.xlabel("Input feature")
plt.ylabel("Target value")
plt.legend()
plt.show()



### **2.3. Precomputed Kernel Matrices**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_02_03.jpg?v=1783999443" width="250">



>* Supply custom pairwise similarities to kernel models
>* Use domain knowledge for complex data types

>* Keep kernel matrix ordering consistent
>* Build test similarities to training examples

>* Powerful for complex, moderate-sized datasets
>* Validate kernels and compare simpler alternatives



In [ ]:
#@title Python Code - Precomputed Kernel Matrices

# This example teaches precomputed kernel matrices.
# Similarities replace ordinary feature columns during fitting.
# The output compares correct and incorrect test shapes.

import numpy as np
import sklearn
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# Small points form two nonlinear classes.
rng = np.random.default_rng(42)
angles = rng.uniform(0.0, 2.0 * np.pi, 80)

inner_radius = rng.normal(1.0, 0.08, 40)
outer_radius = rng.normal(2.0, 0.08, 40)
radii = np.concatenate([inner_radius, outer_radius])

x_values = radii * np.cos(angles)
y_values = radii * np.sin(angles)
features = np.column_stack([x_values, y_values])
labels = np.array([0] * 40 + [1] * 40)

# Split raw observations before computing test similarities.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, stratify=labels, random_state=42
)

# Training kernels are square train-by-train similarity matrices.
train_kernel = rbf_kernel(X_train, X_train, gamma=1.0)
if train_kernel.shape[0] != train_kernel.shape[1]:
    raise ValueError("Training kernel must be square.")

# Test kernels compare each test row with every training row.
test_kernel = rbf_kernel(X_test, X_train, gamma=1.0)
if test_kernel.shape[1] != train_kernel.shape[0]:
    raise ValueError("Test kernel columns must match training rows.")

# The SVM receives similarities, not the original coordinates.
model = SVC(kernel="precomputed")
model.fit(train_kernel, y_train)
predicted_labels = model.predict(test_kernel)

# A common mistake is building a test-by-test kernel.
wrong_test_kernel = rbf_kernel(X_test, X_test, gamma=1.0)
correct_shape = str(test_kernel.shape)
wrong_shape = str(wrong_test_kernel.shape)

accuracy = accuracy_score(y_test, predicted_labels)
print("scikit-learn version:", sklearn.__version__)
print("Training kernel shape:", train_kernel.shape)
print("Correct test kernel shape:", correct_shape)
print("Wrong test-by-test shape:", wrong_shape)
print("Precomputed-kernel test accuracy:", round(accuracy, 3))



## **3. Practical Tradeoffs**

### **3.1. Scale Before Kernels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_03_01.jpg?v=1783999433" width="250">



>* Feature scale strongly affects kernel similarity
>* Scaling prevents large-unit features dominating

>* Distance kernels need balanced feature scales
>* Scaling improves fair accuracy and smoothness comparisons

>* Scaling stabilizes SVR and KRR tuning choices
>* Fair comparisons need scaled kernel inputs



In [ ]:
#@title Python Code - Scale Before Kernels

# This example shows why scaling matters.
# Kernel similarity depends on feature distances.
# Scaling improves fair SVR model comparison.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

# Create two features with very different numeric scales.
rng = np.random.default_rng(42)
samples = 180

small_feature = rng.uniform(0, 1, samples)
large_feature = rng.uniform(0, 1000, samples)

# The target mostly depends on the small-scale feature.
noise = rng.normal(0, 0.08, samples)
y = np.sin(6 * small_feature) + noise

X = np.column_stack([small_feature, large_feature])

# Split before scaling to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

if X_train.shape[1] != 2:
    raise ValueError("Expected exactly two input features.")

# Fit the same RBF SVR with and without scaling.
unscaled_model = SVR(kernel="rbf", C=10, gamma="scale", epsilon=0.05)
scaled_model = make_pipeline(
    StandardScaler(), SVR(kernel="rbf", C=10, gamma="scale", epsilon=0.05)
)

unscaled_model.fit(X_train, y_train)
scaled_model.fit(X_train, y_train)

# Compare test error from the two workflows.
unscaled_rmse = np.sqrt(mean_squared_error(
    y_test, unscaled_model.predict(X_test)
))
scaled_rmse = np.sqrt(mean_squared_error(
    y_test, scaled_model.predict(X_test)
))

print("scikit-learn version:", sklearn.__version__)
print("Unscaled RBF SVR test RMSE:", round(unscaled_rmse, 3))
print("Scaled RBF SVR test RMSE:", round(scaled_rmse, 3))

# Plot predictions while holding the large feature fixed.
grid_small = np.linspace(0, 1, 200)
fixed_large = np.full(200, np.median(large_feature))
grid_X = np.column_stack([grid_small, fixed_large])

plt.figure(figsize=(7, 4))
ax = plt.gca()
ax.scatter(small_feature, y, s=18, alpha=0.45, label="data")
ax.plot(grid_small, unscaled_model.predict(grid_X), label="unscaled SVR")
ax.plot(grid_small, scaled_model.predict(grid_X), label="scaled SVR")
ax.set_title("Scaling changes an RBF kernel regression fit")
ax.set_xlabel("Small-scale feature")
ax.set_ylabel("Target value")
ax.legend()
plt.show()



### **3.2. Computational Complexity**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_03_02.jpg?v=1783999437" width="250">



>* Kernel flexibility increases memory and training cost
>* Large datasets may require simpler alternatives

>* KRR uses all training points
>* SVR speed depends on support vectors

>* Cross-validation can greatly increase kernel costs
>* Plan scalability before full nonlinear modeling



### **3.3. SVR KRR Tradeoffs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_B/image_03_03.jpg?v=1783999435" width="250">



>* SVR ignores small errors using support vectors
>* Kernel Ridge fits all points smoothly

>* Kernel Ridge gives smoother, steadier predictions
>* SVR smoothness depends on careful tuning

>* KRR simpler, but scales poorly with data
>* SVR may predict faster when sparse



In [ ]:
#@title Python Code - SVR KRR Tradeoffs

# Compare SVR and Kernel Ridge tradeoffs.
# Kernels fit smooth nonlinear regression curves.
# Metrics reveal accuracy, sparsity, and cost.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

# A small noisy curve makes the tradeoff visible.
rng = np.random.default_rng(42)
x = np.linspace(0, 6, 120)
y = np.sin(x) + 0.25 * rng.normal(size=x.shape)

# Scikit-learn expects a two-dimensional feature matrix.
X = x.reshape(-1, 1)
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature and target lengths must match.")

# Hold out data to compare predictive accuracy fairly.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Both models use the same RBF kernel idea.
svr_model = make_pipeline(
    StandardScaler(), SVR(kernel="rbf", C=10.0, epsilon=0.15, gamma=0.8)
)

krr_model = make_pipeline(
    StandardScaler(), KernelRidge(kernel="rbf", alpha=0.2, gamma=0.8)
)

# Fit each model once on the same training data.
svr_model.fit(X_train, y_train)
krr_model.fit(X_train, y_train)

# Evaluate accuracy on unseen test data.
svr_mae = mean_absolute_error(y_test, svr_model.predict(X_test))
krr_mae = mean_absolute_error(y_test, krr_model.predict(X_test))

# SVR may predict from fewer support vectors.
svr_step = svr_model.named_steps["svr"]
svr_support_count = len(svr_step.support_)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"SVR test MAE: {svr_mae:.3f}; support vectors: {svr_support_count}/84")
print(f"KRR test MAE: {krr_mae:.3f}; training points used: 84/84")

# Plot fitted curves to compare smoothness visually.
x_grid = np.linspace(0, 6, 300).reshape(-1, 1)
svr_curve = svr_model.predict(x_grid)
krr_curve = krr_model.predict(x_grid)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train[:, 0], y_train, s=22, alpha=0.55, label="training data")
ax.plot(x_grid[:, 0], svr_curve, label="SVR curve", linewidth=2)
ax.plot(x_grid[:, 0], krr_curve, label="Kernel Ridge curve", linewidth=2)
ax.set_title("SVR versus Kernel Ridge on noisy nonlinear data")
ax.set_xlabel("Input feature")
ax.set_ylabel("Target value")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**SVR Kernel Ridge**</font>


In this lecture, you learned to:
- Fit SVR variants for nonlinear and linear regression tasks. 
- Use One-Class SVM and precomputed kernels in appropriate workflows. 
- Compare SVR and Kernel Ridge in accuracy, smoothness, and computational cost. 

In the next Module (Module 13), we will go over 'Neighbors Metrics'